# Exploring Word Embeddings

applying three different types of static word embeddings to the preprocessed IMDB dataset:
1. **Word2Vec**
2. **GloVe**
3. **FastText**


## Import Libraries

In [1]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from gensim.models import Word2Vec, FastText
import gensim.downloader as api


## 1. Load the Preprocessed Data

In [2]:
df = pd.read_csv('IMDB_Dataset_preprocessed_sample.csv')
df.head()


,review,cleaned_review,sentiment
0,I really liked this Summerslam due to the look...,really like summerslam due look arena curtain ...,positive
1,Not many television shows appeal to quite as m...,many television show appeal quite many differe...,positive
2,The film quickly gets to a major chase scene w...,film quickly get major chase scene ever increa...,negative
3,Jane Austen would definitely approve of this o...,jane austen would definitely approve one gwyne...,positive
4,Expectations were somewhat high for me when I ...,expectation somewhat high go see movie think s...,negative


We need to extract the cleaned text and convert each review into a list of words so the models can learn from them.

In [3]:
# tokenize each review into a list of words
corpus = [str(review).split() for review in df['cleaned_review']]

print(f'Total reviews in corpus: {len(corpus)}')
print(f'First review tokens: {corpus[0][:15]}')


Total reviews in corpus: 1000
First review tokens: ['really', 'like', 'summerslam', 'due', 'look', 'arena', 'curtain', 'look', 'overall', 'interesting', 'reason', 'anyways', 'could', 'one', 'good']


## 2. Word2Vec

Word2Vec learns word associations from a large corpus of text. training a Skip-gram model on our IMDB corpus.


In [4]:
w2v_model = Word2Vec(
    sentences=corpus,
    vector_size=100,  # size of the word vectors
    window=5,         # context window size
    min_count=5,      # ignore words that appear less than 5 times
    sg=1              # 1 = skip-gram
)

print(f'Vocabulary size: {len(w2v_model.wv)}')
print(f'Vector for "movie" (first 10 values): {w2v_model.wv["movie"][:10]}...')


Vocabulary size: 3509
Vector for "movie" (first 10 values): [ 0.00740521  0.04208538  0.02345425  0.15625395  0.04579187 -0.21818164
  0.22573021  0.54649675 -0.23217233 -0.09891044]...


Let's look at words similar to 'movie' and 'scary':

In [5]:
w2v_model.wv.most_similar('movie', topn=5)


[('review', 0.9708620309829712),
 ('fan', 0.969176709651947),
 ('horror', 0.9682947397232056),
 ('sure', 0.9676691293716431),
 ('definitely', 0.9632237553596497)]

In [6]:
w2v_model.wv.most_similar('scary', topn=5)


[('cheesy', 0.9953379034996033),
 ('sad', 0.9953174591064453),
 ('appreciate', 0.9953101873397827),
 ('imagine', 0.9952211380004883),
 ('thought', 0.9951379299163818)]

## 3. GloVe (Pre-trained)

Instead of training our own model, we can load a pre-trained GloVe model. This model was trained on Wikipedia and contains a massive vocabulary.


In [7]:
# downloading/loading the pre-trained GloVe vectors
glove_model = api.load('glove-wiki-gigaword-100')

print(f'GloVe vocabulary size: {len(glove_model)}')


GloVe vocabulary size: 400000


Checking similarities using GloVe:

In [8]:
glove_model.most_similar('movie', topn=5)


[('film', 0.9055121541023254),
 ('movies', 0.8959327340126038),
 ('films', 0.866355299949646),
 ('hollywood', 0.8239826560020447),
 ('comedy', 0.8141382932662964)]

In [9]:
glove_model.most_similar('scary', topn=5)


[('frightening', 0.8533830046653748),
 ('weird', 0.8047352433204651),
 ('creepy', 0.7674310207366943),
 ('funny', 0.7536983489990234),
 ('awful', 0.7528400421142578)]

## 4. FastText

FastText is similar to Word2Vec but it breaks words down into character n-grams. This allows it to generate embeddings even for words it hasn't seen before (out-of-vocabulary words).


In [10]:
ft_model = FastText(
    sentences=corpus,
    vector_size=100,
    window=5,
    min_count=5,
    sg=1
)

print(f'Vocabulary size: {len(ft_model.wv)}')


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Vocabulary size: 3509


Similar words in FastText:

In [11]:
ft_model.wv.most_similar('movie', topn=5)


[('laugh', 0.9944850206375122),
 ('worth', 0.993822455406189),
 ('enjoy', 0.9925737380981445),
 ('recommend', 0.9898889660835266),
 ('ugh', 0.988836944103241)]

To test FastText, trying to get a vector for a made-up or rare word that definitely wasn't in the training data:

In [12]:
made_up_word = 'superduperscary'

print(f'Vector for "{made_up_word}": {ft_model.wv[made_up_word][:10]}...')


Vector for "superduperscary": [-0.0452779   0.0924537   0.05460671  0.06154015 -0.0655968   0.07358645
 -0.09527995  0.21113107  0.19779596 -0.09244527]...
